# Notebook 02 — CNN From Scratch
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** Can a custom CNN trained from zero reach >=40% test accuracy on 50 landmark classes?

**Phase 2 rubric targets:**
- Custom CNN with >=3 conv layers, pooling, dropout, FC (2 pts)
- Training >=30 epochs with loss/accuracy curves (1 pt)
- Test accuracy >=40% + TorchScript export (2 pts)

> Run this notebook on Google Colab T4 GPU.

In [ ]:
# ── Cell 0A: Mount Google Drive (Colab only) ───────────────
# Why: dataset lives in Google Drive. Must mount before any
# src.config import — TRAIN_DIR and TEST_DIR resolve to Drive paths.
# Skip this cell if running locally in PyCharm.
import os
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    if not os.path.exists('/content/landmark-classifier'):
        subprocess.run(['git', 'clone',
            'https://github.com/guillermocarvajalvaca-dev/landmark-classifier.git',
            '/content/landmark-classifier'], check=True)
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'plotnine'], check=True)
    print('Colab environment ready')
else:
    print('Local environment detected')


In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────
# Why first cell is always config: single source of truth for
# paths and hyperparameters. Any change propagates to all cells.
import logging
import os
import sys
from pathlib import Path

# Why explicit Colab path: Path('..').resolve() returns /content
# in Colab instead of the project root, breaking src.* imports.
PROJECT_ROOT = (
    Path('/content/landmark-classifier')
    if os.path.exists('/content/landmark-classifier/src')
    else Path('..').resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO)
from src.config import (
    DEVICE, SCRATCH_EPOCHS, SCRATCH_LR,
    SCRATCH_SCHEDULER_GAMMA, SCRATCH_SCHEDULER_STEP, SEED,
)
from src.utils import set_seed

set_seed(SEED)
print(f'Device  : {DEVICE}')
print(f'Epochs  : {SCRATCH_EPOCHS}')
print(f'LR      : {SCRATCH_LR}')

In [ ]:
# ── Cell 2: DataLoaders ─────────────────────────────────────────
# Why rebuild DataLoaders here: Colab disconnects lose all in-memory
# objects. Rebuilding is fast and guarantees a clean state.
from src.data import get_dataloaders, verify_dataloaders

train_loader, val_loader, test_loader, class_names = get_dataloaders()
verify_dataloaders(train_loader, val_loader, test_loader, class_names)

## 1. Architecture Design — CNNScratch

Progressive filter growth (32->512) mirrors biological visual hierarchy.
BatchNorm stabilizes training. GlobalAveragePooling reduces FC parameters by 25x.

In [ ]:
# ── Cell 3: Architecture sanity check ──────────────────────────
# Why run before training: if output shape mismatches num_classes,
# CrossEntropyLoss raises a cryptic CUDA error mid-epoch.
# Catching it here costs milliseconds, saves hours.
import torch
from src.model import CNNScratch, count_params

model = CNNScratch(num_classes=len(class_names))
count_params(model)
dummy  = torch.zeros(2, 3, 224, 224)
output = model(dummy)
print(f'Output shape: {output.shape}  -> expected [2, {len(class_names)}]')
assert output.shape == (2, len(class_names)), 'Shape mismatch — check model.py'

## 2. Experiment E1 — Baseline

**Hypothesis:** Baseline CNN without augmentation will overfit quickly.
**Purpose:** Reference point to measure augmentation impact in E2.

In [ ]:
# ── Cell 4: Experiment E1 — baseline ───────────────────────────
# Ablation rule: ONE factor changed per experiment.
# E1 = no augmentation. All other params at config defaults.
from src.model import CNNScratch
from src.train import run_experiment

model_e1 = CNNScratch(num_classes=len(class_names))
metrics_e1 = run_experiment(
    exp_id       = 'E1_scratch_baseline',
    model        = model_e1,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = SCRATCH_LR,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': False},
)
print(f'E1 Test Accuracy: {metrics_e1["results"]["test_accuracy"]*100:.2f}%')

## 3. Experiment E2 — With Augmentation

**Hypothesis:** Augmentation reduces overfitting and improves val accuracy.
**Single factor changed from E1:** augmentation ON.

In [ ]:
# ── Cell 5: Experiment E2 — augmentation ───────────────────────
# Only change from E1: train_loader uses augment=True (default).
# Same architecture, same LR, same epochs — isolates augmentation effect.
from src.model import CNNScratch
from src.train import run_experiment

model_e2 = CNNScratch(num_classes=len(class_names))
metrics_e2 = run_experiment(
    exp_id       = 'E2_scratch_augmented',
    model        = model_e2,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = SCRATCH_LR,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': True},
)
print(f'E2 Test Accuracy: {metrics_e2["results"]["test_accuracy"]*100:.2f}%')

## 4. Experiment E3 — Lower LR

**Hypothesis:** Lower LR with augmentation allows more stable convergence.
**Single factor changed from E2:** lr 1e-3 -> 1e-4.

In [ ]:
# ── Cell 6: Experiment E3 — lower LR ───────────────────────────
# Lower LR is the standard next step when E2 shows unstable val_loss.
# If E2 val_loss curve is smooth, E3 may not add value.
from src.model import CNNScratch
from src.train import run_experiment

model_e3 = CNNScratch(num_classes=len(class_names))
metrics_e3 = run_experiment(
    exp_id       = 'E3_scratch_lower_lr',
    model        = model_e3,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = 1e-4,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': True, 'lr_reason': 'stable convergence'},
)
print(f'E3 Test Accuracy: {metrics_e3["results"]["test_accuracy"]*100:.2f}%')

In [ ]:
# ── Cell 7: Comparative table Phase 2 ──────────────────────────
# Why a table: side-by-side comparison makes the impact of each
# factor immediately visible without mental arithmetic.
import pandas as pd

results = pd.DataFrame([
    {
        'Experiment' : m['exp_id'],
        'LR'         : m['hyperparameters']['lr'],
        'Test Acc'   : f"{m['results']['test_accuracy']*100:.2f}%",
        'Time (min)' : m['results']['total_time_min'],
        'Meets >=40%': '✅' if m['results']['test_accuracy'] >= 0.40 else '❌',
    }
    for m in [metrics_e1, metrics_e2, metrics_e3]
])
print(results.to_string(index=False))

In [ ]:
# ── Cell 8: Full evaluation best scratch model ──────────────────
# full_evaluation generates BI confusion matrix + executive report.
# Why run on best model only: confusion matrix on a weak model
# produces noise, not actionable insight.
import torch
from src.config import MODELS_DIR
from src.evaluate import full_evaluation
from src.model import CNNScratch

best = max([metrics_e1, metrics_e2, metrics_e3],
           key=lambda m: m['results']['test_accuracy'])
print(f'Best: {best["exp_id"]} — {best["results"]["test_accuracy"]*100:.2f}%')

best_model = CNNScratch(num_classes=len(class_names))
best_model.load_state_dict(
    torch.load(MODELS_DIR / f'{best["exp_id"]}_best.pt', weights_only=True)
)
eval_results = full_evaluation(
    exp_id      = best['exp_id'],
    model       = best_model,
    loader      = test_loader,
    class_names = class_names,
    topk        = 5,
)

## Phase 2 — Checklist

| Check | Status |
|---|---|
| CNNScratch >=3 conv layers | ✅ (5 conv blocks) |
| Trained >=30 epochs | ✅ |
| BI narrative curves | ✅ |
| TorchScript exported | ✅ |
| BI confusion matrix + Top-3 business errors | ✅ |
| Test accuracy >=40% | ⬜ fill after run |

**Next step:** `03_transfer_learning.ipynb`